<a href="https://colab.research.google.com/github/Likitha-sd/Credit_Risk_Modelling/blob/main/Copy_of_MAMBA_CTC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Cell 1 : Environment Setup
# ============================================================

# ---------- Mount Drive ----------
from google.colab import drive
drive.mount('/content/drive')

# ---------- Check GPU ----------
!nvidia-smi

import os
import sys
import warnings
warnings.filterwarnings("ignore")

import torch

print("="*70)
print("Torch Version :", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

print("="*70)

# ---------- Install Dependencies ----------
!pip install -q tifffile scikit-image opencv-python

# ---------- Clone Official Mamba ----------
%cd /content

!rm -rf mamba

!git clone https://github.com/state-spaces/mamba.git

%cd /content/mamba

# ---------- Install Mamba ----------
!pip install -e . --no-build-isolation

# ---------- Verify ----------
try:
    from mamba_ssm import Mamba
    print("\n✅ Official Mamba Installed Successfully!\n")
except Exception as e:
    print(e)

# ---------- Imports ----------
import cv2
import tifffile
import numpy as np
import matplotlib.pyplot as plt

from glob import glob
from pathlib import Path
from PIL import Image

from skimage.measure import regionprops

from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# ============================================================
# DATASET PATH
# ============================================================

ROOT = "/content/drive/MyDrive/DIC-C2DH-HeLa_train"

assert os.path.exists(ROOT), "Dataset not found!"

print("\nDataset Found!\n")

print(os.listdir(ROOT))

# ============================================================
# TRAIN
# ============================================================

SEQ1_IMG = os.path.join(ROOT,"01")

SEQ1_SEG = os.path.join(ROOT,"01_ST","SEG")

SEQ1_TRA = os.path.join(ROOT,"01_GT","TRA")

print("\nSequence 01 Images :",len(glob(os.path.join(SEQ1_IMG,"*.tif"))))
print("Tracking Masks :",len(glob(os.path.join(SEQ1_TRA,"man_track*.tif"))))

print("\nEnvironment Ready.")

Mounted at /content/drive
Fri Jul  3 05:33:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [ ]:
# ============================================================
# DATASET PATHS
# ============================================================

ROOT_TRAIN = "/content/drive/MyDrive/Colab Notebooks/DIC-C2DH-HeLa_train"
ROOT_TEST  = "/content/drive/MyDrive/Colab Notebooks/DIC-C2DH-HeLa_test_"

# ============================================================
# TRAIN
# ============================================================

SEQ1_IMG = os.path.join(ROOT_TRAIN, "01")
SEQ1_SEG = os.path.join(ROOT_TRAIN, "01_ST", "SEG")
SEQ1_TRA = os.path.join(ROOT_TRAIN, "01_GT", "TRA")

SEQ2_IMG = os.path.join(ROOT_TRAIN, "02")
SEQ2_SEG = os.path.join(ROOT_TRAIN, "02_ST", "SEG")
SEQ2_TRA = os.path.join(ROOT_TRAIN, "02_GT", "TRA")

print("="*60)
print("Checking Dataset...")

print("SEQ1 Images :", len(glob(os.path.join(SEQ1_IMG, "*.tif"))))
print("SEQ1 Seg    :", len(glob(os.path.join(SEQ1_SEG, "*.tif"))))
print("SEQ1 Track  :", len(glob(os.path.join(SEQ1_TRA, "man_track*.tif"))))

print()

print("SEQ2 Images :", len(glob(os.path.join(SEQ2_IMG, "*.tif"))))
print("SEQ2 Seg    :", len(glob(os.path.join(SEQ2_SEG, "*.tif"))))
print("SEQ2 Track  :", len(glob(os.path.join(SEQ2_TRA, "man_track*.tif"))))

print("="*60)

In [ ]:
# ============================================================
# CELL 2 : Extract Cells Frame-by-Frame
# ============================================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from glob import glob
from PIL import Image
from skimage.measure import regionprops

# ------------------------------------------------------------
# Read Images
# ------------------------------------------------------------

image_files = sorted(glob(os.path.join(SEQ1_IMG, "*.tif")))[:84]
mask_files  = sorted(glob(os.path.join(SEQ1_SEG, "*.tif")))

print("Images :", len(image_files))
print("Masks  :", len(mask_files))

# ------------------------------------------------------------
# Build Detection Dataset
# ------------------------------------------------------------

IMG_SIZE = 64

detections = []

for frame_id, (img_path, mask_path) in enumerate(zip(image_files, mask_files)):

    image = np.array(Image.open(img_path))
    mask  = np.array(Image.open(mask_path))

    props = regionprops(mask)

    for prop in props:

        label = prop.label

        minr, minc, maxr, maxc = prop.bbox

        crop = image[minr:maxr, minc:maxc]

        crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))

        crop = crop.astype(np.float32) / 255.0

        detections.append({

            "frame": frame_id,

            "label": label,

            "centroid": prop.centroid,

            "bbox": prop.bbox,

            "crop": crop

        })

print("="*60)
print("Frames :", len(image_files))
print("Detections :", len(detections))
print("="*60)

# ------------------------------------------------------------
# Visualization
# ------------------------------------------------------------

plt.figure(figsize=(12,3))

for i in range(6):

    plt.subplot(1,6,i+1)

    plt.imshow(detections[i]["crop"], cmap="gray")

    plt.title(f'F{detections[i]["frame"]}')

    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 3 : CNN Feature Encoder
# ============================================================

import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------------------
# CNN Encoder
# ------------------------------------------------------------

class CellEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv2d(1,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64,128,3,padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1,1))
        )

    def forward(self,x):

        x = self.encoder(x)

        return x.view(x.size(0),-1)

encoder = CellEncoder().to(device)

encoder.eval()

print(encoder)

In [ ]:
# ============================================================
# CELL 4 : Encode Every Detection
# ============================================================

encoded_detections = []

encoder.eval()

with torch.no_grad():

    for det in detections:

        crop = det["crop"]

        crop = torch.tensor(crop).unsqueeze(0).unsqueeze(0).float().to(device)

        feat = encoder(crop).cpu().squeeze(0)

        encoded_detections.append({

            "frame": det["frame"],
            "label": det["label"],
            "centroid": det["centroid"],
            "bbox": det["bbox"],
            "feature": feat

        })

print("="*60)
print("Encoded Detections :", len(encoded_detections))
print("Feature Dimension  :", encoded_detections[0]["feature"].shape)
print("="*60)

In [ ]:
# ============================================================
# CELL 5 : Group Detections by Frame
# ============================================================

from collections import defaultdict

frame_detections = defaultdict(list)

for det in encoded_detections:

    frame_detections[det["frame"]].append(det)

print("="*60)

print("Frames :", len(frame_detections))

for f in range(5):

    print(f"Frame {f} : {len(frame_detections[f])} detections")

print("="*60)

In [ ]:
# ============================================================
# CELL 6 : Improved Hungarian Matching
# ============================================================

import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity

ALPHA = 0.5
BETA = 0.3
GAMMA = 0.2
MAX_COST = 0.22

matches = []

TOTAL_FRAMES = len(frame_detections)

for frame in range(TOTAL_FRAMES - 1):

    det1 = frame_detections[frame]
    det2 = frame_detections[frame + 1]

    if len(det1) == 0 or len(det2) == 0:
        continue

    N = len(det1)
    M = len(det2)

    cost = np.zeros((N, M), dtype=np.float32)

    for i in range(N):

        d1 = det1[i]

        f1 = d1["feature"].numpy()
        c1 = np.array(d1["centroid"])

        h1 = d1["bbox"][2] - d1["bbox"][0]
        w1 = d1["bbox"][3] - d1["bbox"][1]
        area1 = h1 * w1

        for j in range(M):

            d2 = det2[j]

            f2 = d2["feature"].numpy()
            c2 = np.array(d2["centroid"])

            h2 = d2["bbox"][2] - d2["bbox"][0]
            w2 = d2["bbox"][3] - d2["bbox"][1]
            area2 = h2 * w2

            appearance = 1 - cosine_similarity(
                f1.reshape(1, -1),
                f2.reshape(1, -1)
            )[0, 0]

            motion = np.linalg.norm(c1 - c2) / 100.0

            shape = abs(area1 - area2) / (max(area1, area2) + 1e-6)

            cost[i, j] = (
                ALPHA * appearance +
                BETA * motion +
                GAMMA * shape
            )

    rows, cols = linear_sum_assignment(cost)

    for r, c in zip(rows, cols):

        if cost[r, c] < MAX_COST:

            matches.append({
                "frame1": frame,
                "frame2": frame + 1,
                "cell1": det1[r]["label"],
                "cell2": det2[c]["label"],
                "cost": float(cost[r, c])
            })

print("=" * 60)
print("Matched Pairs :", len(matches))
print("=" * 60)

for m in matches[:10]:
    print(
        f'Frame {m["frame1"]} Cell {m["cell1"]} '
        f'--> Frame {m["frame2"]} Cell {m["cell2"]} '
        f'Cost={m["cost"]:.4f}'
    )

In [ ]:
# ============================================================
# CELL 7 : Improved Track Management
# ============================================================

import numpy as np

tracks = {}
active_tracks = {}

next_track_id = 0

MAX_DISTANCE = 25      # pixels
MAX_MISSING = 2        # keep track alive

# ------------------------------------------------------------
# Initialize Frame 0
# ------------------------------------------------------------

for det in frame_detections[0]:

    tracks[next_track_id] = [det]

    active_tracks[next_track_id] = {
        "last_frame": 0,
        "last_centroid": np.array(det["centroid"]),
        "label": det["label"]
    }

    next_track_id += 1

# ------------------------------------------------------------
# Process Remaining Frames
# ------------------------------------------------------------

for frame in range(1, len(frame_detections)):

    matched_tracks = set()
    matched_cells = set()

    # -------------------------
    # Hungarian Matches
    # -------------------------

    for m in matches:

        if m["frame2"] != frame:
            continue

        cell1 = m["cell1"]
        cell2 = m["cell2"]

        track_id = None

        for tid, info in active_tracks.items():

            if info["label"] == cell1:

                track_id = tid
                break

        if track_id is None:
            continue

        det = next(
            d for d in frame_detections[frame]
            if d["label"] == cell2
        )

        tracks[track_id].append(det)

        active_tracks[track_id] = {

            "last_frame": frame,
            "last_centroid": np.array(det["centroid"]),
            "label": cell2

        }

        matched_tracks.add(track_id)
        matched_cells.add(cell2)

    # -------------------------
    # Handle Unmatched Cells
    # -------------------------

    for det in frame_detections[frame]:

        if det["label"] in matched_cells:
            continue

        centroid = np.array(det["centroid"])

        best_track = None
        best_distance = 1e9

        for tid, info in active_tracks.items():

            if frame - info["last_frame"] > MAX_MISSING:
                continue

            dist = np.linalg.norm(
                centroid - info["last_centroid"]
            )

            if dist < best_distance:

                best_distance = dist
                best_track = tid

        # Continue nearby track

        if best_track is not None and best_distance < MAX_DISTANCE:

            tracks[best_track].append(det)

            active_tracks[best_track] = {

                "last_frame": frame,
                "last_centroid": centroid,
                "label": det["label"]

            }

        # Otherwise create new track

        else:

            tracks[next_track_id] = [det]

            active_tracks[next_track_id] = {

                "last_frame": frame,
                "last_centroid": centroid,
                "label": det["label"]

            }

            next_track_id += 1

print("="*60)

print("Predicted Tracks :", len(tracks))

lengths = [len(v) for v in tracks.values()]

print("Average Length :", np.mean(lengths))
print("Longest Track :", np.max(lengths))
print("Shortest Track :", np.min(lengths))

print("="*60)

In [ ]:
print(len(tracks))

In [ ]:
# ============================================================
# CELL 8 : Build Tracklet Sequences (NEW)
# ============================================================

import torch

track_sequences = {}

for tid, history in tracks.items():

    features = []

    for det in history:

        features.append(det["feature"])

    if len(features) >= 2:

        track_sequences[tid] = torch.stack(features)

print("="*60)
print("Tracklet Sequences :", len(track_sequences))

first_track = next(iter(track_sequences))

print("Example Track :", first_track)
print("Sequence Shape :", track_sequences[first_track].shape)

print("="*60)

In [ ]:
single = 0
multi = 0

for tid, history in tracks.items():

    if len(history) == 1:
        single += 1
    else:
        multi += 1

print("Single-frame tracks :", single)
print("Multi-frame tracks :", multi)

In [ ]:
# ============================================================
# CELL 9 : Predictive Temporal Mamba
# ============================================================

import torch
import torch.nn as nn
from mamba_ssm import Mamba

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class PredictiveTemporalMamba(nn.Module):

    def __init__(self):

        super().__init__()

        self.mamba = Mamba(
            d_model=128,
            d_state=16,
            d_conv=4,
            expand=2
        )

        # Predict next appearance feature
        self.feature_head = nn.Sequential(
            nn.Linear(128,128),
            nn.ReLU(),
            nn.Linear(128,128)
        )

    def forward(self,x):

        # x : (B,T,128)

        h = self.mamba(x)

        last = h[:,-1]          # last timestep

        next_feature = self.feature_head(last)

        return next_feature


model = PredictiveTemporalMamba().to(device)

print(model)

In [ ]:
# ============================================================
# CELL 10 : Refine Track Embeddings
# ============================================================

refined_tracks = {}

model.eval()

with torch.no_grad():

    for tid, sequence in track_sequences.items():

        sequence = sequence.unsqueeze(0).to(device)

        emb = model.encode(sequence)

        refined_tracks[tid] = emb.squeeze(0).cpu()

print("="*60)
print("Refined Tracks :", len(refined_tracks))

first_track = next(iter(refined_tracks))

print("Embedding Shape :", refined_tracks[first_track].shape)
print("="*60)

In [ ]:
# ============================================================
# CELL 11 : Verify Refined Track Embeddings
# ============================================================

print("="*60)

print("Temporal Tracklets :", len(track_sequences))

print("Refined Embeddings :", len(refined_tracks))

first = next(iter(refined_tracks))

print("Embedding Shape :", refined_tracks[first].shape)

print("="*60)

In [ ]:
# ============================================================
# CELL 12 : Create Next-Frame Prediction Dataset
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader

WINDOW = 5

samples = []

for tid, seq in track_sequences.items():

    if len(seq) < WINDOW:
        continue

    for i in range(len(seq) - WINDOW + 1):

        history = seq[i:i+WINDOW-1]     # Frames 1-4

        target  = seq[i+WINDOW-1]       # Frame 5

        samples.append((history, target))

print("="*60)
print("Training Samples :", len(samples))
print("="*60)


class TemporalPredictionDataset(Dataset):

    def __init__(self, samples):

        self.samples = samples

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        x, y = self.samples[idx]

        return x.float(), y.float()


prediction_dataset = TemporalPredictionDataset(samples)

prediction_loader = DataLoader(
    prediction_dataset,
    batch_size=16,
    shuffle=True
)

x, y = next(iter(prediction_loader))

print("Input Shape :", x.shape)
print("Target Shape:", y.shape)

In [ ]:
# ============================================================
# CELL 13 : Final Refined Embeddings
# ============================================================

model.eval()

final_embeddings = {}

with torch.no_grad():

    for tid, seq in track_sequences.items():

        seq = seq.unsqueeze(0).to(device)

        emb = model.encode(seq)

        final_embeddings[tid] = emb.squeeze(0).cpu()

print("="*60)

print("Final Embeddings :", len(final_embeddings))

first_track = next(iter(final_embeddings))

print("Embedding Shape :", final_embeddings[first_track].shape)

print("="*60)

In [ ]:
# ============================================================
# CELL 14 : Similarity Matrix
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import numpy as np

embeddings = np.stack([
    emb.numpy()
    for emb in final_embeddings.values()
])

track_ids = list(final_embeddings.keys())

sim = cosine_similarity(embeddings)

plt.figure(figsize=(8,8))

plt.imshow(sim, cmap="viridis")

plt.colorbar()

plt.xticks(
    np.arange(len(track_ids)),
    track_ids,
    rotation=90
)

plt.yticks(
    np.arange(len(track_ids)),
    track_ids
)

plt.title("Temporal Mamba Similarity Matrix")

plt.show()

In [ ]:
# ============================================================
# CELL 15 : t-SNE Visualization
# ============================================================

from sklearn.manifold import TSNE

tsne = TSNE(
    n_components=2,
    perplexity=min(5, len(track_ids)-1),
    random_state=42
)

proj = tsne.fit_transform(embeddings)

plt.figure(figsize=(8,8))

for i in range(len(track_ids)):

    plt.scatter(
        proj[i,0],
        proj[i,1],
        s=120
    )

    plt.text(
        proj[i,0],
        proj[i,1],
        str(track_ids[i]),
        fontsize=10
    )

plt.title("Temporal Mamba Embeddings")

plt.show()

In [ ]:
# ============================================================
# CELL 16 : Compare Predicted vs Ground Truth
# ============================================================

track_txt = os.path.join(SEQ1_TRA, "man_track.txt")

gt_tracks = {}

with open(track_txt) as f:

    for line in f:

        tid, start, end, parent = map(int, line.split())

        gt_tracks[tid] = {

            "start": start,
            "end": end,
            "parent": parent

        }

print("="*60)

print("Ground Truth Tracks :", len(gt_tracks))
print("Predicted Tracks    :", len(tracks))

print("="*60)

common = min(len(gt_tracks), len(tracks))

print(f"Coverage : {common}/{len(gt_tracks)}")

In [ ]:
# ============================================================
# CELL 17 : Track Duration Comparison
# ============================================================

gt_lengths = []

for tid, info in gt_tracks.items():

    gt_lengths.append(
        info["end"] - info["start"] + 1
    )

pred_lengths = [
    len(history)
    for history in tracks.values()
]

print("="*60)

print("Ground Truth")
print(" Tracks :", len(gt_lengths))
print(" Avg Length :", np.mean(gt_lengths))
print(" Max Length :", np.max(gt_lengths))
print(" Min Length :", np.min(gt_lengths))

print()

print("Predicted")
print(" Tracks :", len(pred_lengths))
print(" Avg Length :", np.mean(pred_lengths))
print(" Max Length :", np.max(pred_lengths))
print(" Min Length :", np.min(pred_lengths))

print("="*60)

In [ ]:
# ============================================================
# CELL 18 : Ground Truth vs Predicted Comparison
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

metrics = [
    "Tracks",
    "Avg Length",
    "Max Length",
    "Min Length"
]

gt = [
    38,
    np.mean(gt_lengths),
    np.max(gt_lengths),
    np.min(gt_lengths)
]

pred = [
    len(tracks),
    np.mean(pred_lengths),
    np.max(pred_lengths),
    np.min(pred_lengths)
]

x = np.arange(len(metrics))

width = 0.35

plt.figure(figsize=(8,5))

plt.bar(
    x-width/2,
    gt,
    width,
    label="Ground Truth"
)

plt.bar(
    x+width/2,
    pred,
    width,
    label="Predicted"
)

plt.xticks(x, metrics)

plt.ylabel("Value")

plt.title("Ground Truth vs Predicted Tracking Statistics")

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# CELL 20 : Detection Precision / Recall / F1
# ============================================================

import numpy as np

TP = 0
FP = 0
FN = 0

TOTAL_FRAMES = min(len(frame_detections), len(track_masks))

for frame in range(TOTAL_FRAMES):

    # Ground Truth cells
    gt = np.array(Image.open(track_masks[frame]))
    gt_ids = set(np.unique(gt))
    gt_ids.discard(0)

    # Predicted cells
    pred_ids = set()

    if frame in frame_detections:

        for det in frame_detections[frame]:

            pred_ids.add(det["label"])

    TP += len(gt_ids & pred_ids)

    FP += len(pred_ids - gt_ids)

    FN += len(gt_ids - pred_ids)

precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print("="*60)

print(f"True Positives  : {TP}")
print(f"False Positives : {FP}")
print(f"False Negatives : {FN}")

print()

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("="*60)

In [ ]:
# ============================================================
# CELL 21 : Tracking Evaluation
# ============================================================

import numpy as np

id_switches = 0
fragmentations = 0
continuous_tracks = 0

track_lengths = []

for tid, history in tracks.items():

    frames = [d["frame"] for d in history]

    track_lengths.append(len(frames))

    # --------------------------------------------------------
    # Check continuity
    # --------------------------------------------------------

    gaps = np.diff(sorted(frames))

    if np.all(gaps == 1):

        continuous_tracks += 1

    else:

        fragmentations += 1

    # --------------------------------------------------------
    # Count ID switches
    # --------------------------------------------------------

    labels = [d["label"] for d in history]

    for i in range(1, len(labels)):

        if labels[i] != labels[i-1]:

            id_switches += 1

print("="*60)

print("Tracking Evaluation")

print()

print("Predicted Tracks :", len(tracks))
print("Continuous Tracks:", continuous_tracks)
print("Fragmented Tracks:", fragmentations)
print("ID Switches      :", id_switches)

print()

print("Average Track Length :", np.mean(track_lengths))
print("Maximum Track Length :", np.max(track_lengths))
print("Minimum Track Length :", np.min(track_lengths))

print("="*60)

In [ ]:
# ============================================================
# CELL 22 : Approximate MOTA
# ============================================================

GT = TP + FN

MOTA = 1 - ((FP + FN + id_switches) / GT)

print("="*60)
print(f"Approximate MOTA : {MOTA:.4f}")
print("="*60)

In [ ]:
# ============================================================
# CELL 22 : Approximate MOTA
# ============================================================

GT = TP + FN

MOTA = 1 - ((FP + FN + id_switches) / GT)

print("="*60)
print(f"Approximate MOTA : {MOTA:.4f}")
print("="*60)

In [ ]:
# ============================================================
# CELL 23 : Cell Trajectory (X-Y)
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(10,10))

for tid, history in tracks.items():

    xs = [d["centroid"][1] for d in history]   # X
    ys = [d["centroid"][0] for d in history]   # Y

    plt.plot(xs, ys, linewidth=2, label=f"Cell {tid}")

    plt.scatter(xs[0], ys[0], s=60, marker="o")     # Start
    plt.scatter(xs[-1], ys[-1], s=80, marker="x")   # End

plt.gca().invert_yaxis()

plt.xlabel("X Position (pixels)", fontsize=12)
plt.ylabel("Y Position (pixels)", fontsize=12)

plt.title("Predicted Cell Trajectories", fontsize=15)

plt.grid(True)

plt.legend(bbox_to_anchor=(1.05,1), loc="upper left")

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# Find Track Variables
# ============================================================

for name in globals():

    if "track" in name.lower():

        obj = globals()[name]

        print(name, type(obj))

        try:
            print("Length:", len(obj))
        except:
            pass

        print("-"*50)